# 분석 테이블(데이터마트) 만들기

`EDA.ipynb`(1부 전처리 · 2부 EDA)에서 정한 설계대로 **기대 모델에 넣을 표 하나**를 만들어
`data/mart.parquet` 으로 저장함. 만들어진 표는 `Modeling.ipynb`(기대 모델 피팅)에서 읽어 씀.

- 설계 근거: `docs/설계 문서` 3절 · 연구계획서(비공개) 6~7장
- 커널: conda 환경 **impok** (Python 3.11)

## 만드는 표

**한 행 = 유저 1명 × 기준일 1일.** 1,781만 행 / 유저 110만 명 / 설명 변수 23개.

| | |
|---|---|
| 기준일 | 음악을 들은 날. 3/1~3/30 (3/31은 결과일이 4/1이라 못 씀) |
| 결과 변수 | `d1` — 기준일 다음 날에도 들었나 (0/1) |
| 설명 변수 | **전부 기준일 t 까지 관측 가능한 값만.** 결과일 정보·`is_churn` 은 쓰지 않음 |

## 왜 이 표가 필요한가

리텐션이 떨어졌을 때 "어느 집단 때문인가"를 말하려면, 집단마다 **원래 얼마였어야 하는지(기대값)**
가 있어야 함. 집단마다 평소 복귀율이 다르기 때문임(신규는 낮고 오래된 유료 구독자는 높음).
이 표로 **정상 기간에 기대 모델을 학습**해 두고, 평가 기간에 그 모델을 대어
**편차 = 실제 − 기대**를 구함. 편차가 크게 음수인 집단이 가설 후보가 됨.

## 절 구성

| 절 | 내용 | 만드는 것 |
|---|---|---|
| 0 | 준비 | 라이브러리·상수 |
| 1 | 데이터 불러오기 | `members`, `user_logs`, `user_logs_feb` |
| 2 | 3월 활동 기록을 정수로 바꿔 저장 (캐시) | `code`, `day`, `key_all`, `uniq` |
| 3 | 결과 변수 | `u`, `day`, `d1` |
| 4 | **A. 시간 구조** | `days_since_signup`, `base_dow` |
| 5 | **B. 유저 특성** | `city`, `age`, `gender`, `channel`, (`profile`) |
| 6 | **C. 구독 상태** | `has_tx` 외 8개 |
| 7 | **D. 평소 습관** | `act_days` 외 7개 |
| 8 | 저장 | `data/mart.parquet` |
| 9 | 최종 변수 설명 · 결측과 이상치 처리 | (문서) |


## 0. 준비

In [19]:
import numpy as np
import pandas as pd

In [20]:
DATA = 'data'
BASE_LAST = 29          # 기준일 마지막 = 3/30 (3/31은 결과일이 4/1이라 D1 불가)
TRAIN_LAST = 14         # 학습 기간 마지막 기준일 = 3/15

## 1. 데이터 불러오기

세 파일을 읽음. 거래(`transactions_all.parquet`)는 무거워서 **6절에서 그때 읽음**.

| 파일 | 쓰는 곳 | 행 수 |
|---|---|---|
| `members_v3.parquet` | 4절 A군(가입일) · 5절 B군(도시·나이·성별·경로) | 677만 |
| `user_logs_v2.parquet` | 2절 — 2017년 3월 청취 기록 (분석 기간) | 1,840만 |
| `user_logs_feb2017.parquet` | 7절 D군 — 2017년 2월 청취 기록 (습관 측정 창) | 1,608만 |

습관을 **2월에 재는 이유**: 기준일마다 직전 N일을 다시 재면, 이벤트로 생긴 행동 감소까지
"평소 모습"으로 보게 되어 **찾아야 할 이상을 모델이 흡수함** (계획서 표 6).

In [21]:
members = pd.read_parquet('data/members_v3.parquet')
user_logs = pd.read_parquet('data/user_logs_v2.parquet')          # 2017년 3월 = 분석 기간
user_logs_feb = pd.read_parquet('data/user_logs_feb2017.parquet')  # 2017년 2월 = 평소 습관 측정 창

## 2. 3월 활동 기록을 정수로 바꿔 저장 (캐시)

"이 유저가 그날 음악을 들었나"를 **1,781만 번** 물어야 함. 44글자 문자열(`msno`)로 비교하거나
표를 합치면(merge) 중간에 거대한 표가 생겨 램이 터짐. 그래서 **(유저, 날짜)를 정수 하나로 눌러 담고 정렬**해 둠.

    key = 유저번호 x 64 + 날짜번호        (3/1 = 0 … 3/31 = 30)

64를 곱하는 건 유저 자리와 날짜 자리가 섞이지 않게 **자리를 떼어 두는 것**임
(`20170301` 이 년x10000 + 월x100 + 일 인 것과 같은 발상). 날짜 자리를 64칸으로 넉넉히 잡은 건
D7을 볼 때 `day + 7` = 최대 37 까지 조회하기 때문임.

만드는 데 1~2분 걸리고 결과는 늘 같으므로 `data/cache/` 에 저장해 두고 다음부터는 읽어 씀
(데이터가 바뀌면 행 수가 달라지므로 `n` 값으로 낡았는지 판단함).

| 만드는 것 | 길이 | 뜻 |
|---|---|---|
| `code` | 1,840만 | 그 줄이 몇 번 유저인가 |
| `day` | 1,840만 | 그 줄이 며칠인가 (3/1 = 0) |
| `key_all` | 1,840만 | 활동한 (유저, 날)의 정수 키를 **정렬**해 둔 것 |
| `uniq` | 110만 | 유저 번호 → 원래 `msno` 대응표 |

In [22]:
import os

# 연습용으로 다른 이름에 저장함 — 진짜 캐시(active_keys.npz)는 건드리지 않음
OUT       = f'{DATA}/cache/active_keys_test.npz'
OUT_USERS = f'{DATA}/cache/active_users_test.parquet'
os.makedirs(f'{DATA}/cache', exist_ok=True)

# ── 1) 원본 로그에서 쓸 두 컬럼만 읽음 ────────────────────────────────────
# 1,839만 줄임. num_25 같은 다른 컬럼은 지금 필요 없어 columns= 로 걸러 읽음
user_logs = user_logs [['msno', 'date']]
print('원본 로그 {:,}줄'.format(len(user_logs)))


원본 로그 18,396,362줄


In [23]:
# ── 2) msno(44글자 문자열)를 유저 번호로 바꿈 ─────────────────────────────
# factorize는 두 개를 돌려줌 — 줄마다의 번호(code), 번호→msno 대응표(uniq)
# 번호는 '처음 나온 순서'대로 0, 1, 2… 로 붙음
code, uniq = pd.factorize(user_logs['msno'])
print('유저 {:,}명'.format(len(uniq)))

# ── 3) date(20170301)를 날짜 번호로 바꿈 (3/1 = 0 … 3/31 = 30) ────────────
# 전부 3월이라 '일' 자리만 떼면 됨.  20170301 % 100 = 1  →  -1  →  0
day = user_logs['date'].to_numpy() % 100 - 1

# ── 4) (유저, 날짜)를 숫자 하나로 눌러 담고 정렬함 ────────────────────────
key_all = np.sort(code.astype('int64') * 64 + day)
code, day = code.astype('int32'), day.astype('int8')

# ── 6) 파일로 저장함 ──────────────────────────────────────────────────────
# n은 '만들 때 원본이 몇 줄이었나' 메모임 — 다음에 캐시가 낡았는지 판단하는 데 씀
np.savez(OUT, code=code, day=day, key_all=key_all, n=np.array([len(user_logs)]))
pd.Series(uniq, name='msno').to_frame().to_parquet(OUT_USERS, compression='zstd', index=False)

print('\n저장 완료')
print('  code    ', code.dtype, code.shape)
print('  day     ', day.dtype, day.shape)
print('  key_all ', key_all.dtype, key_all.shape)

유저 1,103,894명

저장 완료
  code     int32 (18396362,)
  day      int8 (18396362,)
  key_all  int64 (18396362,)


In [24]:
# ── 1) 길이·범위·정렬 ────────────────────────────────────────────────────
assert len(code) == len(day) == len(key_all) == len(user_logs), '길이가 안 맞음'
assert day.min() == 0 and day.max() == 30, '3월은 31일이라 0~30이어야 함'
assert code.min() == 0 and code.max() == len(uniq) - 1, '유저 번호가 0부터 연속이어야 함'
assert (np.diff(key_all) >= 0).all(), 'key_all이 정렬돼 있지 않음'
print('검증 1: 길이·범위·정렬 통과')

# ── 2) 진짜 캐시와 똑같이 나왔나 — 가장 확실한 검증임 ─────────────────────
_z0 = np.load(f'{DATA}/cache/active_keys.npz')
print('\n진짜 캐시와 비교:')
print('  code    :', np.array_equal(code, _z0['code']))
print('  day     :', np.array_equal(day, _z0['day']))
print('  key_all :', np.array_equal(key_all, _z0['key_all']))
_u0 = pd.read_parquet(f'{DATA}/cache/active_users.parquet')['msno']
print('  uniq    :', _u0.equals(pd.Series(uniq, name='msno')))

# ── 3) 0번 유저로 손 검증 — d1이 어떻게 만들어지는지 눈으로 봄 ────────────
_d0 = np.sort(day[code == 0])
print('\n0번 유저가 들은 날짜 번호:', _d0)
for _d in _d0[:5]:
    _k  = 0 * 64 + int(_d)          # 그날 들었나
    _k2 = 0 * 64 + int(_d) + 1      # 다음 날에도 들었나  ← 이게 d1
    _i  = np.searchsorted(key_all, _k)
    _i2 = min(np.searchsorted(key_all, _k2), len(key_all) - 1)
    print('  day {:2d} → 키 {:3d} 있나 {}   |  다음날 키 {:3d} 있나 {}  ← d1'.format(
        _d, _k, key_all[_i] == _k, _k2, key_all[_i2] == _k2))

# ── 4) 램 비우기 ─────────────────────────────────────────────────────────
del user_logs
print('\nuser_logs 비움')

검증 1: 길이·범위·정렬 통과

진짜 캐시와 비교:
  code    : True
  day     : True
  key_all : True
  uniq    : True

0번 유저가 들은 날짜 번호: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30]
  day  0 → 키   0 있나 True   |  다음날 키   1 있나 True  ← d1
  day  1 → 키   1 있나 True   |  다음날 키   2 있나 True  ← d1
  day  2 → 키   2 있나 True   |  다음날 키   3 있나 True  ← d1
  day  3 → 키   3 있나 True   |  다음날 키   4 있나 True  ← d1
  day  4 → 키   4 있나 True   |  다음날 키   5 있나 True  ← d1

user_logs 비움


## 3. 결과 변수 `d1` — 기준일 행 만들기

기준일마다 **다음 날에도 들었는지**를 구함. `(유저, 기준일+1)` 의 키를 만들어
활동 목록(`key_all`)에 있는지 보면 됨.

| 만드는 것 | 뜻 |
|---|---|
| `u` | 유저 번호 |
| `day` | 기준일 번호 (0 = 3/1 … 29 = 3/30) |
| `d1` | **결과 변수.** 기준일 다음 날에도 들었나 (True/False) |

**3/31은 기준일에서 뺌** — 결과일이 4/1인데 데이터에 4월이 없음. 그냥 두면 "안 돌아왔다"가
아니라 "모른다"인 것을 0으로 세게 되어, 3/31의 복귀율이 0%라는 거짓이 생김.

⚠️ `d1` 은 **미래(t+1) 정보가 맞음** — 이건 누수가 아니라 **정답**임. 누수 금지는 설명 변수에만
적용됨. 정상 기간에 "보통 얼마나 돌아오는지"를 재야 기대값의 기준(자)을 만들 수 있음.

In [25]:
# ── 1) '다음 날에도 들었나'를 구함 ────────────────────────────────────────
# 각 줄의 (유저, 기준일+1)을 숫자 키로 적고, 그 키가 활동 목록(key_all)에 있나 봄
k_next = code.astype('int64') * 64 + (day + 1)
hit_next = np.isin(k_next, key_all)

# ── 2) 기준일 행만 남김 ──────────────────────────────────────────────────
# 3/31은 결과일이 4/1인데 데이터에 4월이 없음 → '안 돌아왔다'가 아니라 '모른다'라서 뺌
rows_ok = day <= BASE_LAST
base = pd.DataFrame({'u'  : code[rows_ok],
                     'day': day[rows_ok].astype('int16'),
                     'd1' : hit_next[rows_ok]})

print('기준일 행 {:,}  /  유저 {:,}명'.format(len(base), base['u'].nunique()))
print('전체 D1 {:.4f}'.format(base['d1'].mean()))
base.head(10)

기준일 행 17,818,632  /  유저 1,099,987명
전체 D1 0.7816


,u,day,d1
0,1,29,False
1,6,29,True
2,11,23,True
3,12,24,False
4,13,23,True
5,14,24,True
6,15,24,True
7,16,23,True
8,17,24,True
9,18,17,True


In [26]:
# ── 1) project_1.ipynb 에서 이미 나온 숫자와 맞춰 봄 ─────────────────────
print('유저 수            {:,}      (기대 1,103,894)'.format(base['u'].nunique()))
print('3/1~3/14 D1        {:.4f}   (기대 0.7782  ← 9절 exact_return(1))'.format(
    base.loc[base['day'] <= 13, 'd1'].mean()))
print('학습 3/1~3/15  D1  {:.4f}   (기대 0.7796  ← 12절)'.format(
    base.loc[base['day'] <= TRAIN_LAST, 'd1'].mean()))
print('평가 3/16~3/30 D1  {:.4f}   (기대 0.7836  ← 12절)'.format(
    base.loc[base['day'] > TRAIN_LAST, 'd1'].mean()))

# ── 2) 날짜별 D1 — 요일에 따라 오르내리는지 눈으로 봄 ────────────────────
_daily = base.groupby('day')['d1'].agg(분모='size', D1='mean')
_daily.index = pd.Timestamp('2017-03-01') + pd.to_timedelta(_daily.index, unit='D')
_daily['요일'] = _daily.index.day_name().str[:3]
print('\n기준일별 D1:')
print(_daily.assign(D1=lambda d: (d['D1'] * 100).round(2)).to_string())

# ── 3) 0번 유저 손 검증 — 표의 d1 이 실제 활동일과 맞나 ──────────────────
_mine = base[base['u'] == 0].sort_values('day')
_days0 = np.sort(day[code == 0])
print('\n0번 유저가 들은 날짜 번호:', _days0)
print(_mine.assign(직접확인=lambda d: [(x + 1) in _days0 for x in d['day']]).to_string(index=False))

유저 수            1,099,987      (기대 1,103,894)
3/1~3/14 D1        0.7782   (기대 0.7782  ← 9절 exact_return(1))
학습 3/1~3/15  D1  0.7796   (기대 0.7796  ← 12절)
평가 3/16~3/30 D1  0.7836   (기대 0.7836  ← 12절)

기준일별 D1:
                분모     D1   요일
day                           
2017-03-01  575889  79.43  Wed
2017-03-02  580623  79.26  Thu
2017-03-03  590594  75.16  Fri
2017-03-04  584124  76.69  Sat
2017-03-05  589177  74.50  Sun
2017-03-06  577955  79.06  Mon
2017-03-07  579079  79.33  Tue
2017-03-08  581288  79.62  Wed
2017-03-09  584919  79.41  Thu
2017-03-10  597915  75.61  Fri
2017-03-11  592497  77.28  Sat
2017-03-12  598768  75.03  Sun
2017-03-13  587823  79.43  Mon
2017-03-14  589627  79.92  Tue
2017-03-15  592706  79.90  Wed
2017-03-16  595837  79.93  Thu
2017-03-17  608931  75.61  Fri
2017-03-18  597723  77.41  Sat
2017-03-19  604289  76.04  Sun
2017-03-20  602846  80.01  Mon
2017-03-21  603087  80.40  Tue
2017-03-22  604399  80.50  Wed
2017-03-23  606033  79.63  Thu
2017-03-24  60847

## 4. A. 시간 구조

리텐션은 **가입한 지 얼마나 됐는가**와 **무슨 요일인가**에 크게 좌우됨. 이걸 기대 모델이
흡수해야 집단별 편차가 그 효과로 오염되지 않음.

| 만드는 변수 | 계산 | 왜 필요한가 |
|---|---|---|
| `days_since_signup` | 기준일 − 가입일 (`registration_init_time`) | **필수.** 3월 신규 집단의 D1이 학습 0.551 → 평가 0.639 로 오르는데, 이건 개선이 아니라 **생존 효과**(초반 이탈자는 월말 기준일 행에 안 들어옴). 빼면 "신규 유저"가 가짜 가설로 올라옴 |
| `base_dow` | 기준일 요일 (0 = 월 … 6 = 일). 3/1이 수요일이라 `(day + 2) % 7` | 금·토·일 기준일 D1이 75~77%, 월~목이 79~81%로 차이가 큼 |

| `days_since_first_mar` | 기준일 − 그 유저의 **3월 첫 활동일** | **실험용(기본 변수 아님).** 10/7에 확인한 생존 효과 때문임 — 2월에 가볍게 들은 유저 중 3월 말까지 남은 사람은 유독 충성도가 높은데, 습관(D)은 2월 스냅샷이라 이를 모름. 편차가 날마다 기울어지는 원인임 |

⚠️ **`days_since_first_mar` 는 표에만 두고 `FEATURES` 에는 넣지 않음.** 넣을지 말지는 주입 실험으로
정해야 하는데, 두 가지 위험이 있기 때문임.

| 위험 | 수치 (10/7 측정) |
|---|---|
| **달력 날짜를 복원함** | 3/1부터 활동한 유저는 이 값이 기준일 번호와 **같음**. 전체 행의 **72.3%** 가 그러함 |
| **외삽을 못 함** | 학습 기간에서는 0~14 까지만 나오는데 평가 기간 행의 **90.9%** 가 14를 넘음. 트리는 본 적 없는 값을 "14와 같다"로 처리함 |

다만 2월 활동이 적은 집단에서는 이 값이 달력 날짜와 달라(미측정 집단은 6.4%만 일치, 중앙값 4일)
**고쳐야 할 집단에서만 쓸모가 있을** 가능성이 있음. 그래서 표에는 만들어 두고 실험으로 판단함.

**넣지 않는 것 두 가지**

- **결과일 요일** — D1에서는 `base_dow + 1` 로 완전히 정해져 정보가 0임
- **가입 코호트(연·월)** — `days_since_signup` 을 한 달 단위로 뭉갠 것이라 중복이고, 더 중요하게는
  `기준일 = 가입 후 경과일 + 가입일` 이라 둘을 함께 넣으면 모델이 **달력 날짜를 ±15일 정확도로 복원**함.
  그러면 "3월 하순은 낮다"를 외워서 찾아야 할 하락을 삼킴 (APC 식별 문제, 계획서 4.2 각주 1)

In [27]:
mem = members.set_index('msno').reindex(uniq)
print('members에 없는 유저 {:,}명 (문서 기준 40명)'.format(
    mem['registration_init_time'].isna().sum()))

# ── 2) 가입일을 '3/1로부터 며칠'로 바꿈 ──────────────────────────────────
# 대부분 3/1보다 전에 가입했으므로 음수가 나옴 (예: 2016-03-01 → -365)
_reg = pd.to_datetime(mem['registration_init_time'].astype('Int64').astype('string'),
                      format='%Y%m%d', errors='coerce')
reg_off = (_reg - pd.Timestamp('2017-03-01')).dt.days.to_numpy().astype('float32')

# ── 3) 기준일 행에 A군 변수를 붙임 ───────────────────────────────────────
_u, _d = base['u'].to_numpy(), base['day'].to_numpy()
base['days_since_signup'] = _d - reg_off[_u]            # 기준일 − 가입일
base['base_dow'] = ((_d + 2) % 7).astype('int8')        # 3/1이 수요일 → 0=월 … 2=수

# ── 4) 3월 첫 활동일 이후 경과일 (실험용, FEATURES 에는 안 넣음) ─────────
# 기준일 행이 곧 활동일이므로 유저별 최솟값이 '3월 첫 활동일'임.
# 누수 없음 — 첫 활동일은 언제나 기준일 이하라 기준일 t 까지 알 수 있는 정보임
_first = np.full(len(uniq), 127, dtype='int16')
np.minimum.at(_first, _u, _d.astype('int16'))
base['days_since_first_mar'] = (_d - _first[_u]).astype('int16')

base.head()

members에 없는 유저 40명 (문서 기준 40명)


,u,day,d1,days_since_signup,base_dow,days_since_first_mar
0,1,29,False,247.0,3,29
1,6,29,True,806.0,3,19
2,11,23,True,120.0,4,23
3,12,24,False,4007.0,5,24
4,13,23,True,500.0,4,18


In [28]:
#── 1) 값의 범위 ─────────────────────────────────────────────────────────
print('가입 후 경과일  최소 {:.0f}  최대 {:.0f}'.format(
    np.nanmin(base['days_since_signup']), np.nanmax(base['days_since_signup'])))
print('결측 {:,}행 ({:.4%})  ← members에 없는 40명 때문임'.format(
    base['days_since_signup'].isna().sum(), base['days_since_signup'].isna().mean()))

# ── 1-2) 3월 첫 활동일 이후 경과일 — 누수와 달력 복원 정도를 함께 봄 ────
_e = base['days_since_first_mar'].to_numpy()
assert (_e >= 0).all() and (_e <= base['day'].to_numpy()).all(), '첫 활동일이 기준일보다 뒤임 — 누수!'
print('\n3월 첫 활동일 이후 경과일  범위 {} ~ {}  (누수 점검 통과)'.format(_e.min(), _e.max()))
print('  기준일 번호와 같은 행 {:.1%}  ← 3/1부터 활동한 유저. 72.3% 가 기대값'.format(
    (_e == base['day'].to_numpy()).mean()))

# ── 2) 요일이 맞게 붙었나 — 3/1은 수요일 ─────────────────────────────────
_c = base.groupby('base_dow')['d1'].agg(행='size', D1='mean')
_c.index = ['월', '화', '수', '목', '금', '토', '일']
print('\n요일별 (금·토·일이 낮아야 함):')
print(_c.to_string(float_format='{:.4f}'.format))

# ── 3) 가입 경과 집단 — 12절 표와 맞춰 봄 ────────────────────────────────
_t = pd.cut(reg_off[_u], [-np.inf, -1095, -365, 0, np.inf], right=False,
            labels=['3 yr +', '1-3 yr', '< 1 yr', 'new (Mar 2017)'])
_g = base.groupby(_t, observed=True)['d1'].agg(행='size', D1='mean')
print('\n가입 경과별 (기대: 신규 32.7만행 D1 0.604 / 1년미만 340만 / 1~3년 532만 / 3년+ 877만):')
print(_g.to_string(float_format='{:.4f}'.format))

가입 후 경과일  최소 0  최대 4752
결측 401행 (0.0023%)  ← members에 없는 40명 때문임

3월 첫 활동일 이후 경과일  범위 0 ~ 29  (누수 점검 통과)
  기준일 번호와 같은 행 72.3%  ← 3/1부터 활동한 유저. 72.3% 가 기대값

요일별 (금·토·일이 낮아야 함):
         행     D1
월  2361494 0.7976
화  2371205 0.8006
수  2956790 0.8004
목  2975152 0.7887
금  2405918 0.7536
토  2364585 0.7706
일  2383488 0.7541

가입 경과별 (기대: 신규 32.7만행 D1 0.604 / 1년미만 340만 / 1~3년 532만 / 3년+ 877만):
                      행     D1
3 yr +          8764807 0.7883
1-3 yr          5314699 0.7892
< 1 yr          3411677 0.7698
new (Mar 2017)   327048 0.6035


## 5. B. 유저 특성

`members` 에서 오는 인구통계·가입 경로. **설명력은 약하지만**(단일 변수로 D1을 4.4~8.9%p만 가름)
집단을 해석하는 축으로 필요함.

| 만드는 변수 | 값 | 비고 |
|---|---|---|
| `city` | `unknown (1)` / 상위 5개 도시 / `other` / `unknown` | 1번이 71%라 "정보 없음"으로 봄 |
| `age` | `10s` `20s` `30s` `40+` `unknown` | 10~80세 밖(0·음수·2016 등)은 전부 `unknown` |
| `gender` | `male` `female` `unknown` | 65%가 빈칸 |
| `channel` | `3` `4` `7` `9` / `other` / `unknown` | 가입 경로 코드 |
| (`profile`) | `complete` / `missing` / `unknown` | **파생 변수** — 위 셋에서 자동 계산됨. 대조용으로 표에는 두되 `FEATURES` 에서는 뺌 |

⚠️ **B군 단일 변수로 해석하면 안 됨** — 신규 코호트에서는 가입 경로가 Day 7을 **62.1%p** 가르는데
전체 유저 D1에서는 5.6%p 뿐이고, **방향까지 뒤집힘**(경로 7번이 코호트 1위인데 전체에선 꼴찌).
"프로필 미입력이 나쁘다"도 성립하지 않음 — 경로별 미입력 비율이 24%~83%로 달라 두 효과를 뗄 수 없음 (8-2절).

**문자열 대신 `Categorical` 로 넣음** — 1,781만 행에 문자열을 그대로 넣으면 램이 수 GB로 불어남.

In [29]:
# ── 1) 유저 단위로 B군 라벨을 만듦 (mem은 이미 유저 번호 순서임) ──────────
# 결측은 지우지 않고 '미입력'이라는 하나의 값으로 둠 (7-3절 결론)
_bd, _city = mem['bd'], mem['city']
_top5 = _city[_city != 1].value_counts().head(5).index          # 1번을 뺀 상위 5개 도시

prof = pd.DataFrame(index=np.arange(len(uniq)))

# 도시 — 1번은 71%라 '정보 없음'으로 봄. 상위 5개만 남기고 나머지는 묶음
prof['city'] = np.where(_city.isna(), 'unknown',
                np.where(_city == 1, 'unknown (1)',
                 np.where(_city.isin(_top5), 'city ' + _city.astype('Int64').astype(str), 'other')))

# 나이 — 10~80세 밖(0, 음수, 2016 같은 값)은 전부 '미입력'
prof['age'] = np.select([~_bd.between(10, 80), _bd < 20, _bd < 30, _bd < 40],
                        ['unknown', '10s', '20s', '30s'], '40+')

prof['gender'] = mem['gender'].fillna('unknown').to_numpy()

# 가입 경로 — 3·4·7·9번이 대부분이라 그것만 남기고 나머지는 묶음
prof['channel'] = mem['registered_via'].map(
    lambda v: str(int(v)) if v in (3, 4, 7, 9) else ('unknown' if pd.isna(v) else 'other')).to_numpy()

# 프로필 입력 여부 — 성별·나이·도시 중 하나라도 비면 '미입력' (7-3절 기준)
prof['profile'] = np.where(mem['registered_via'].isna(), 'unknown',
                   np.where(mem['gender'].isna() | ~_bd.between(10, 80) | (_city == 1),
                            'missing', 'complete'))

# ── 2) 기준일 행에 붙임 ──────────────────────────────────────────────────
# Categorical로 넣음 — 1,782만 행에 문자열을 그대로 넣으면 램이 몇 GB로 불어남
_u = base['u'].to_numpy()
for c in ['city', 'age', 'gender', 'channel', 'profile']:
    base[c] = pd.Categorical(prof[c].to_numpy()[_u])

print(base.dtypes.to_string())
base.head()

u                          int32
day                        int16
d1                          bool
days_since_signup        float32
base_dow                    int8
days_since_first_mar       int16
city                    category
age                     category
gender                  category
channel                 category
profile                 category


,u,day,d1,days_since_signup,base_dow,days_since_first_mar,city,age,gender,channel,profile
0,1,29,False,247.0,3,29,unknown (1),unknown,unknown,7,missing
1,6,29,True,806.0,3,19,city 5,10s,male,3,complete
2,11,23,True,120.0,4,23,unknown (1),unknown,unknown,7,missing
3,12,24,False,4007.0,5,24,city 22,30s,female,9,complete
4,13,23,True,500.0,4,18,city 15,30s,male,9,complete


In [30]:
# ── 1) 12절 표와 대조 — 프로필 입력 여부 ─────────────────────────────────
# 기대: complete 944만행 D1 0.802  /  missing 838만행 D1 0.758
print(base.groupby('profile', observed=True)['d1'].agg(행='size', D1='mean')
      .to_string(float_format='{:.4f}'.format))

# ── 2) 8-2절 표와 대조 — 전체 유저에서 각 특성의 스프레드(%p) ────────────
# 기대: 연령대 8.9 / 도시 5.8 / 가입경로 5.6 / 성별 4.7 / 프로필 4.4
print('\n특성별 스프레드 (가장 높은 칸 D1 − 가장 낮은 칸 D1):')
for c, 기대 in [('age', 8.9), ('city', 5.8), ('channel', 5.6), ('gender', 4.7), ('profile', 4.4)]:
    _g = base.groupby(c, observed=True)['d1'].agg(['size', 'mean'])
    _g = _g[_g['size'] >= 20000]
    print('  {:<8s} {:.1f}%p   (기대 {:.1f})'.format(c, (_g['mean'].max() - _g['mean'].min()) * 100, 기대))

# ── 3) 각 특성의 분포 ────────────────────────────────────────────────────
for c in ['city', 'age', 'gender', 'channel']:
    _g = base.groupby(c, observed=True)['d1'].agg(행='size', D1='mean')
    _g['비중'] = _g['행'] / len(base)
    print('\n[{}]'.format(c))
    print(_g.sort_values('행', ascending=False).to_string(float_format='{:.4f}'.format))

                행     D1
profile                 
complete  9435759 0.8024
missing   8382472 0.7583
unknown       401 0.7506

특성별 스프레드 (가장 높은 칸 D1 − 가장 낮은 칸 D1):
  age      8.9%p   (기대 8.9)
  city     5.8%p   (기대 5.8)
  channel  5.6%p   (기대 5.6)
  gender   4.7%p   (기대 4.7)
  profile  4.4%p   (기대 4.4)

[city]
                   행     D1     비중
city                              
unknown (1)  6954765 0.7497 0.3903
other        2954019 0.7979 0.1658
city 13      2626271 0.8079 0.1474
city 5       1874734 0.7999 0.1052
city 4       1241890 0.8034 0.0697
city 15      1107833 0.8029 0.0622
city 22      1058719 0.8006 0.0594
unknown          401 0.7506 0.0000

[age]
               행     D1     비중
age                           
unknown  7765730 0.7551 0.4358
20s      5380338 0.8167 0.3020
30s      2631736 0.7680 0.1477
40+      1106133 0.7769 0.0621
10s       934695 0.8442 0.0525

[gender]
               행     D1     비중
gender                        
unknown  7752560 0.7553 0.4351
male     5238

## 6. C. 구독 상태

**행마다 "기준일 t 이하의 마지막 거래" 한 건을 찾아 그 정보를 붙였음.**

구독 상태는 날마다 변함 — 갱신하면 요금제·만료일이 바뀌고, 취소하면 자동연장이 꺼짐.
그래서 같은 유저라도 **기준일이 지나가면서 붙는 거래가 달라짐**.

| `u` | 기준일 | 붙은 거래 | `days_to_expire` |
|---|---|---|---|
| 민수 | 3/2 | 2/10 결제 (만료 3/12) | 10 |
| 민수 | 3/5 | 2/10 결제 | 7 |
| 민수 | 3/13 | **3/12 결제** (만료 4/11) ← 갱신해서 바뀜 | 29 |

3/2 행에는 **3/12 거래가 붙지 않음** — 그게 누수 방지의 핵심임.
실제로 10/6에 8-1절에서 같은 종류의 누수를 고쳤음: 결제 그룹을 기간 제한 없이 "첫 결제"로
나눠서, 가입 **후에** 결제한 2,943명이 Day 1~3 리텐션에도 `유료` 로 들어가 있었음.
"유료라서 잘 남았다"가 아니라 **"잘 남은 사람이 나중에 결제했다"** 였음.

### `has_tx` 는 "거래 기록이 있나"지 "지금 구독 중인가"가 아님

마지막 거래의 만료일이 기준일보다 **앞설 수 있음** — 결제했다가 구독이 끊긴 상태임.
이때도 그 거래 정보를 그대로 붙이고 `days_to_expire` 가 **음수**가 됨.
"만료됐으니 무료 유저로 치자"고 바꾸면 **"예전에는 결제했던 사람"** 이라는 정보가 사라지므로
그렇게 하지 않았음. 그래서 세 집단이 구분됨.

| 집단 | `has_tx` | `days_to_expire` | 행 수 |
|---|---|---|---|
| 한 번도 결제 안 함 (진짜 무료) | **False** | 결측 | 270,325 (1.52%) |
| 결제했고 **구독 중** | True | **양수** | 약 1,735만 |
| 결제했으나 **이미 만료** | True | **음수** | 197,487 (1.11%) |

⚠️ **해석 주의** — 만료가 지난 행에서 `payment_method_id` · `is_auto_renew` · `payment_plan_days` ·
`plan_list_price` 는 "지금 쓰는 요금제"가 아니라 **"마지막으로 결제했던 조건"** 임.
가설이 "결제수단 41번 집단이 하락"으로 나오면 그중 일부는 이미 떠난 사람일 수 있으므로
`days_to_expire` 의 부호를 같이 봐야 함. `is_cancel = 1` 이면서 음수면 **해지하고 무료로 듣는 중**임.

기준일까지 거래가 한 건도 없으면 `has_tx = False`, C 변수는 전부 결측이 됨.
이건 **진짜 무료 유저**를 뜻함(270,325행, D1 0.492). 10/5에 `transactions.csv`(v1)를 병합하기
전에는 이 값이 "구독 상태 불명"이라 정상 기간 48%에 달했고 월중에 급변해 가짜 가설을
만들 수 있었음. 병합 후 1.5%로 줄고 의미가 깨끗해짐.

### 만드는 변수

| 변수 | 뜻 |
|---|---|
| `has_tx` | 기준일까지 거래 기록이 있나. **False = 진짜 무료 유저** |
| `payment_method_id` | 결제수단 코드 (**범주로 둠** — 숫자로 두면 "15번보다 작은" 같은 뜻 없는 분할이 생김) |
| `payment_plan_days` | 요금제 기간(일). 7·30·90·180 등 순서가 의미 있어 숫자 그대로 |
| `plan_list_price` / `actual_amount_paid` | 정가 / 실제 결제액 |
| `discount` | 정가 − 실제 결제액 |
| `is_auto_renew` / `is_cancel` | 자동연장 여부 / 그 거래가 취소였나 |
| `days_to_expire` | 구독 만료일 − 기준일. **음수면 이미 만료된 상태** |

### 구현 메모

1,781만 행 x 2,298만 거래를 merge하면 램이 터지므로, 거래를 **(유저, 거래일) 순으로 정렬한
정수 키**로 바꿔 두고 `np.searchsorted` 로 기준일 이하의 마지막 자리를 찾음. 거래가 없는
유저를 조회하면 **앞 번호 유저의 거래**가 잡히므로 유저 번호가 같은지 확인해야 함.
마지막에 `assert` 로 붙은 거래가 전부 기준일 이하인지 **누수 점검**을 함.

In [31]:
# ── 0) YYYYMMDD 정수를 '3/1로부터 며칠'로 바꾸는 도우미 ──────────────────
def ymd_to_offset(v, origin='2017-03-01'):
    """날짜 종류가 수백 개뿐이라 고유값만 변환한 뒤 되돌려 붙임
    (2,298만 개를 전부 문자열로 바꾸면 느리고 램도 많이 먹음)"""
    u = np.unique(v)
    off = (pd.to_datetime(pd.Series(u).astype(str), format='%Y%m%d', errors='coerce')
           - pd.Timestamp(origin)).dt.days.to_numpy().astype('float64')   # ← 결측을 넣을 수 있게 실수로
    return off[np.searchsorted(u, v)]


# ── 1) 거래를 읽고 유저 번호·날짜 번호로 바꿈 ───────────────────────────
CCOLS = ['payment_method_id', 'payment_plan_days', 'plan_list_price',
         'actual_amount_paid', 'is_auto_renew', 'is_cancel']
tx = pd.read_parquet(f'{DATA}/transactions_all.parquet',
                     columns=['msno', 'transaction_date', 'membership_expire_date'] + CCOLS)
print('거래 {:,}건'.format(len(tx)))

_tc = pd.Categorical(tx['msno'], categories=uniq).codes.astype('int32')   # -1 = 3월 활동 유저가 아님
_in = _tc >= 0
print('그중 3월 활동 유저의 거래 {:,}건 ({:.1%})'.format(_in.sum(), _in.mean()))

_tday = ymd_to_offset(tx['transaction_date'].to_numpy())
assert not np.isnan(_tday).any(), '거래일에 날짜가 아닌 값이 있음'
_eday = ymd_to_offset(tx['membership_expire_date'].to_numpy())
# 2000년 이전 만료일(19700101 등 9,717건, 0.042%)은 '정보 없음'을 0으로 채운 흔적이라 결측 처리함
_eday[tx['membership_expire_date'].to_numpy() < 20000101] = np.nan

_vals = {c: tx[c].to_numpy()[_in] for c in CCOLS}      # 쓸 값만 떼어 두고
del tx                                                  # 큰 표는 바로 비움
_tc, _tday, _eday = _tc[_in], _tday[_in].astype('int32'), _eday[_in]

# ── 2) (유저, 거래일) 순으로 세우고 정수 키를 만듦 ──────────────────────
# 거래일이 2015-01-01(= -790)까지 내려가므로 800을 더해 음수를 없앰. 날짜 자리는 1024칸을 뗌
SHIFT, MUL = 800, 1024
assert _tday.min() + SHIFT >= 0 and _tday.max() + SHIFT < MUL, '날짜 자리가 모자람'
_order = np.lexsort((_tday, _tc))                       # 유저 번호 → 거래일 순
tx_key = _tc[_order].astype('int64') * MUL + (_tday[_order] + SHIFT)
assert (np.diff(tx_key) >= 0).all(), '정렬이 안 됨'

# ── 3) 행마다 '기준일 이하의 마지막 거래'를 찾음 ────────────────────────
# side='right' 로 "그 키보다 큰 첫 자리"를 받고 1을 빼면 → 기준일 이하의 마지막 거래가 됨
# (isin 으로는 못 함. '있나'가 아니라 '없으면 그 앞의 것'을 찾는 질문이라 searchsorted 가 맞음)
_u, _d = base['u'].to_numpy(), base['day'].to_numpy()
q = _u.astype('int64') * MUL + (_d + SHIFT)
pos = np.searchsorted(tx_key, q, side='right') - 1
_safe = np.maximum(pos, 0)
# 찾은 자리가 '다른 유저의 마지막 거래'일 수 있으므로 유저가 같은지 확인함
has_tx = (pos >= 0) & (tx_key[_safe] // MUL == _u)
_idx = _order[_safe]                                    # 원래 거래 행 번호

# ── 4) 누수 점검 — 붙은 거래가 전부 기준일 이하인가 ─────────────────────
assert (_tday[_idx][has_tx] <= _d[has_tx]).all(), '기준일 이후 거래가 붙음 — 누수!'
print('누수 점검 통과 — 붙은 거래가 모두 기준일 이하임')

# ── 5) 기준일 행에 붙임 ─────────────────────────────────────────────────
# has_tx 가 False면 '기준일까지 거래 기록 없음' = 진짜 무료 유저 (v1 병합으로 의미가 깨끗해짐)
base['has_tx'] = has_tx
for c in CCOLS:
    _v = _vals[c][_idx].astype('float32')
    _v[~has_tx] = np.nan
    base[c] = _v

base['discount'] = base['plan_list_price'] - base['actual_amount_paid']   # 할인액
_e = _eday[_idx].astype('float32')
_e[~has_tx] = np.nan
base['days_to_expire'] = _e - _d                                          # 만료일 − 기준일

# 결제수단은 코드 값이라 범주로 둠 (숫자로 두면 '15번보다 작은' 같은 뜻 없는 분할이 생김).
#
# ⚠️ 문자열 범주로 만드는 이유 — pyarrow 는 '숫자 범주형'을 parquet 에 쓸 때 범주를 풀어
#    (int16/double 로) 저장해 버려서, 다시 읽으면 범주형이 아니게 됨. 문자열 범주만
#    dictionary 형으로 저장돼 왕복에서 살아남음 (10/7 확인).
# ⚠️ 범주 순서를 '숫자 순'으로 못 박는 이유 — LightGBM 은 범주를 순서로 매긴 코드로 다루므로,
#    순서가 바뀌면 이미 학습해 둔 모델(data/model/)의 분할 기준과 어긋남.
#    문자열 정렬에 맡기면 '10' < '2' 가 되어 순서가 틀어짐
_pm   = base['payment_method_id']
_cats = [str(v) for v in np.sort(_pm.dropna().unique().astype('int32'))]      # 숫자 순
base['payment_method_id'] = pd.Categorical(
    np.where(_pm.notna(), _pm.fillna(0).astype('int32').astype(str), None), categories=_cats)

_c = base['payment_method_id']
print('payment_method_id dtype = {} / 범주 {}개 / 결측 {:,}'.format(
    _c.dtype, len(_c.cat.categories), _c.isna().sum()))
print('  범주 앞 8개:', list(_c.cat.categories[:8]))

print('\n기준일까지 거래 기록이 있는 행 {:.4f}   (기대 약 0.985)'.format(has_tx.mean()))
base.head()

거래 22,975,416건
그중 3월 활동 유저의 거래 14,509,058건 (63.2%)
누수 점검 통과 — 붙은 거래가 모두 기준일 이하임
payment_method_id dtype = category / 범주 36개 / 결측 270,325
  범주 앞 8개: ['2', '3', '5', '6', '8', '10', '11', '12']

기준일까지 거래 기록이 있는 행 0.9848   (기대 약 0.985)


,u,day,d1,days_since_signup,base_dow,days_since_first_mar,city,age,gender,channel,profile,has_tx,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,is_cancel,discount,days_to_expire
0,1,29,False,247.0,3,29,unknown (1),unknown,unknown,7,missing,True,41,30.0,99.0,99.0,1.0,0.0,0.0,27.0
1,6,29,True,806.0,3,19,city 5,10s,male,3,complete,True,38,30.0,149.0,149.0,0.0,0.0,0.0,11.0
2,11,23,True,120.0,4,23,unknown (1),unknown,unknown,7,missing,True,41,30.0,129.0,129.0,1.0,0.0,0.0,30.0
3,12,24,False,4007.0,5,24,city 22,30s,female,9,complete,True,32,410.0,1788.0,1788.0,0.0,0.0,0.0,25.0
4,13,23,True,500.0,4,18,city 15,30s,male,9,complete,True,32,410.0,1788.0,1788.0,0.0,0.0,0.0,326.0


In [32]:
# ── 1) 12절 표와 대조 ────────────────────────────────────────────────────
# 기대: 기록 있음 1,755만행 D1 0.786  /  기록 없음 27만행 D1 0.492
print(base.groupby('has_tx', observed=True)['d1'].agg(행='size', D1='mean')
      .to_string(float_format='{:.4f}'.format))

# ── 2) 구간별 비중 — 드리프트가 해소됐나 (10/5 병합 결과) ───────────────
# 기대: 거래 있음 학습 0.9818 → 평가 0.9855
_p = np.where(base['day'] <= TRAIN_LAST, '(1) 학습', '(2) 평가')
print('\n구간별 거래 기록 보유 비중:')
print(pd.crosstab(_p, base['has_tx'], normalize='index').to_string(float_format='{:.4f}'.format))

# ── 3) 만료까지 남은 일수 — 범위와 결측 ─────────────────────────────────
_x = base['days_to_expire']
print('\n만료까지 남은 일수  최소 {:.0f}  중앙 {:.0f}  최대 {:.0f}'.format(
    _x.min(), _x.median(), _x.max()))
print('  결측 {:,}행 ({:.4%})  ← 거래 없음 + 만료일 2000년 이전'.format(_x.isna().sum(), _x.isna().mean()))
print('  음수(이미 만료) {:,}행 ({:.4%})'.format((_x < 0).sum(), (_x < 0).mean()))

# ── 4) C 변수별 D1 ──────────────────────────────────────────────────────
for c in ['is_auto_renew', 'is_cancel']:
    print('\n[{}]'.format(c))
    print(base.groupby(c, observed=True, dropna=False)['d1'].agg(행='size', D1='mean')
          .to_string(float_format='{:.4f}'.format))

print('\n[payment_method_id] 상위 8개')
_g = base.groupby('payment_method_id', observed=True)['d1'].agg(행='size', D1='mean')
print(_g.sort_values('행', ascending=False).head(8).to_string(float_format='{:.4f}'.format))

               행     D1
has_tx                 
False     270325 0.4917
True    17548307 0.7861

구간별 거래 기록 보유 비중:
has_tx  False  True 
row_0               
(1) 학습 0.0171 0.9829
(2) 평가 0.0133 0.9867

만료까지 남은 일수  최소 -813  중앙 21  최대 3690
  결측 270,428행 (1.5177%)  ← 거래 없음 + 만료일 2000년 이전
  음수(이미 만료) 197,487행 (1.1083%)

[is_auto_renew]
                      행     D1
is_auto_renew                 
0.0             5808968 0.8186
1.0            11739339 0.7700
NaN              270325 0.4917

[is_cancel]
                  행     D1
is_cancel                 
0.0        17403997 0.7865
1.0          144310 0.7312
NaN          270325 0.4917

[payment_method_id] 상위 8개
                         행     D1
payment_method_id                
41                 6294749 0.7500
38                 2283268 0.8324
32                 1889229 0.8057
36                 1486618 0.7973
40                 1363315 0.7906
39                 1159768 0.7929
37                  684461 0.8014
34                  401919 0.7787

## 7. D. 평소 습관 (2월 한 달, 고정 측정)

**가장 강력한 변수 묶음임** — 활동일 수 하나로 D1이 48.6%p 갈림(결제 기록 29.4%p, B군은 4~9%p).

측정 창은 **2/1~2/28 (정확히 4주)** 이고, 유저당 한 번만 재서 모든 기준일 행에 같은 값을 붙임.
기준일마다 직전 7일을 다시 재는 방식은 쓰지 않음 — 이벤트로 생긴 행동 감소까지 "평소"로 보게 되어
**찾아야 할 이상을 모델이 흡수**하기 때문임 (계획서 표 6).

| 만드는 변수 | 축 | 계산 | 스프레드 |
|---|---|---|---|
| `act_days` | 빈도 | 2월 활동일 수 (0~28) | **48.6%p** |
| `mins_per_day` | 양 | 활동한 날의 평균 청취 시간(분) | 23.8%p |
| `days_since_last` | 최근성 | 2/28 − 2월 마지막 활동일 (0~27) | 22.8%p |
| `weekend_ratio` | 시간 패턴 | 토·일 활동일 ÷ 전체 활동일 | 21.9%p |
| `trend` | 추세 | 2/15~2/28 활동일 − 2/1~2/14 활동일 (−14~+14) | 20.2%p |
| `pct_unique` | 다양성 | 서로 다른 곡 수 ÷ 전체 곡 수 | 9.2%p |
| `pct_full` | 질 | 끝까지 들은 곡 수 ÷ 전체 곡 수 | 5.2%p |
| `no_habit` | — | 2월에 활동이 없어 측정 불가 (0/1) | — |

**왜 2월 28일 창인가** — 처음 설계는 3/1~3/7 7일이었는데, "7일 내내 들음"에 **35%가 뭉쳐**
천장 효과가 심했음. 28일로 늘리면 그 뭉친 집단이 D1 0.851 → 0.973 으로 갈림.
또 `trend` 와 `weekend_ratio` 는 **28일 창이라야 계산 가능**함.

⚠️ `trend` 해석 주의 — 2월이 **설 연휴 직후**라 DAU가 월 내내 9.6% 오름. 그래서 음수는
"활동이 급감한 집단"이 아니라 **"연휴 이후 평소 수준으로 돌아오지 않은 집단"** 으로 읽어야 함.

⚠️ `no_habit` 플래그가 **필수**임 — 결측으로는 집단 탐색 조건을 만들 수 없고, 또 이 집단의 D1이
활동일이 적은 집단보다 **오히려 높아서** 연속 변수의 끝 값으로 채우면 안 됨.

In [33]:
# ── 1) 2월 로그에서 유저별 습관을 집계함 ─────────────────────────────────
# 습관은 '한 번 재서 고정'하는 값임. 기준일마다 직전 N일을 다시 재면 이벤트로 생긴
# 행동 감소까지 '평소 모습'으로 보게 되어, 찾아야 할 이상을 모델이 흡수함 (계획서 표 6)
_h  = user_logs_feb
_hc = pd.Categorical(_h['msno'], categories=uniq).codes.astype('int32')  # 유저 번호 (-1 = 3월에 활동 안 한 유저)
_hd = (_h['date'].to_numpy() % 100 - 1).astype('int8')                   # 2/1 → 0 … 2/28 → 27
_in = _hc >= 0                                                           # 3월에도 활동한 유저만 씀
print('2월 로그 {:,}행 중 {:,}행({:.1%})을 씀'.format(len(_h), _in.sum(), _in.mean()))

# total_secs 이상값 — 하루 24시간(86,400초)을 넘는 행은 ②를 계산할 때만 뺌
_ts    = _h['total_secs'].to_numpy()
_songs = (_h['num_25'] + _h['num_50'] + _h['num_75'] + _h['num_985'] + _h['num_100']).to_numpy()
_n100, _nunq = _h['num_100'].to_numpy(), _h['num_unq'].to_numpy()
del _h                                              # 이름만 지움. user_logs_feb 자체는 그대로 둠
_neg, _over = _ts < 0, _ts > 86400
print('total_secs  음수 {:,}행  /  86,400초 초과 {:,}행({:.4%})'.format(
    _neg.sum(), _over.sum(), _over.mean()))
_hc, _hd, _ts, _songs, _n100, _nunq = (a[_in] for a in (_hc, _hd, _ts, _songs, _n100, _nunq))
_ok = ~(_neg | _over)[_in]

# ── 2) 유저 번호별 합계 (np.bincount = groupby보다 가볍고 결과는 같음) ───
_n  = len(uniq)
_bc = lambda w=None, f=None: np.bincount(_hc if f is None else _hc[f], minlength=_n,
                                         weights=w if f is None else (None if w is None else w[f]))
_days     = _bc().astype('int16')             # 2월 활동일 수 (하루 한 줄이라 줄 수 = 날 수)
_vd       = _bc(None, _ok)                    # 이상값을 뺀 날 수
_secs     = _bc(_ts.astype('float64'), _ok)
_sg       = _bc(_songs.astype('float64'))
_f100     = _bc(_n100.astype('float64'))
_funq     = _bc(_nunq.astype('float64'))
_last     = np.full(_n, -1, 'int8'); np.maximum.at(_last, _hc, _hd)   # 2월 마지막 활동일
_first14  = _bc(None, _hd <= 13)              # 2/1~2/14 활동일 수
_second14 = _bc(None, _hd >= 14)              # 2/15~2/28 활동일 수
# 2017-02-01이 수요일 → 날짜번호 % 7 이 3이면 토, 4면 일. 2월은 정확히 4주라 주말이 8일임
_wkend    = _bc(None, np.isin(_hd % 7, [3, 4]))
_has      = _days > 0                         # 측정 창에 활동한 유저

# ── 3) 습관 표 (한 행 = 유저 1명) ────────────────────────────────────────
HABIT_COLS = ['act_days', 'mins_per_day', 'pct_full', 'pct_unique',
              'days_since_last', 'trend', 'weekend_ratio']
habit = pd.DataFrame(index=np.arange(_n))
habit['act_days'] = _days                                                              # ① 빈도
with np.errstate(invalid='ignore', divide='ignore'):
    habit['mins_per_day'] = np.where(_vd > 0, _secs / np.maximum(_vd, 1) / 60, np.nan)  # ② 양
    habit['pct_full']     = np.where(_sg > 0, _f100 / np.maximum(_sg, 1), np.nan)       # ③ 질
    habit['pct_unique']   = np.where(_sg > 0, _funq / np.maximum(_sg, 1), np.nan)       # ④ 다양성
habit['days_since_last']  = np.where(_has, 27 - _last, np.nan)                          # ⑤ 최근성
habit['trend']            = np.where(_has, _second14 - _first14, np.nan)                # ⑥ 추세
habit['weekend_ratio']    = np.where(_has, _wkend / np.maximum(_days, 1), np.nan)       # ⑦ 시간 패턴
habit.loc[~_has, ['mins_per_day', 'pct_full', 'pct_unique']] = np.nan
# 측정 불가 플래그 — 결측으로는 집단 탐색 조건을 만들 수 없어 따로 둠. 또 이 집단의 D1이
# 활동일이 적은 집단보다 오히려 높아서, 연속 변수의 끝 값으로 채우면 안 됨
habit['no_habit'] = ~_has

# ── 4) 기준일 행에 붙임 — 같은 유저는 모든 기준일 행에서 같은 값 ─────────
_u = base['u'].to_numpy()
for c in HABIT_COLS + ['no_habit']:
    _v = habit[c].to_numpy()[_u]
    base[c] = _v.astype('float32') if _v.dtype.kind == 'f' else _v    # float64면 1GB 가까이 먹음

print('\n측정 창(2월) 활동 유저 {:,} / {:,} ({:.1%})'.format(_has.sum(), _n, _has.mean()))
base.head()

2월 로그 16,079,619행 중 15,579,565행(96.9%)을 씀
total_secs  음수 0행  /  86,400초 초과 3,820행(0.0238%)

측정 창(2월) 활동 유저 954,116 / 1,103,894 (86.4%)


,u,day,d1,days_since_signup,base_dow,days_since_first_mar,city,age,gender,channel,...,discount,days_to_expire,act_days,mins_per_day,pct_full,pct_unique,days_since_last,trend,weekend_ratio,no_habit
0,1,29,False,247.0,3,29,unknown (1),unknown,unknown,7,...,0.0,27.0,15,28.585682,0.548023,0.875706,0.0,7.0,0.266667,False
1,6,29,True,806.0,3,19,city 5,10s,male,3,...,0.0,11.0,1,4.402500,1.000000,1.000000,27.0,-1.0,0.000000,False
2,11,23,True,120.0,4,23,unknown (1),unknown,unknown,7,...,0.0,30.0,26,63.244934,0.432945,0.908163,0.0,2.0,0.269231,False
3,12,24,False,4007.0,5,24,city 22,30s,female,9,...,0.0,25.0,23,80.506844,0.781879,0.879195,0.0,-5.0,0.347826,False
4,13,23,True,500.0,4,18,city 15,30s,male,9,...,0.0,326.0,15,31.884153,0.644172,0.815951,0.0,5.0,0.266667,False


In [34]:
# ── 1) 13절 수치와 대조 ──────────────────────────────────────────────────
print('2월 로그 중 사용 행 비중 {:.3f}   (기대 0.969)'.format(_in.mean()))
print('측정 창 활동 유저 비중   {:.3f}   (기대 0.864)'.format(_has.mean()))
print('기준일 행 중 측정 불가   {:.4f}  (기대 0.0376)'.format(base['no_habit'].mean()))

# ── 2) ① 활동일 수별 D1 — 양 끝이 문서와 맞나 ───────────────────────────
_g = base.groupby('act_days', observed=True)['d1'].agg(행='size', D1='mean')
print('\n활동일 수별 D1 (기대: 1일 0.494 → 28일 0.968):')
print(_g.loc[[0, 1, 4, 7, 14, 21, 27, 28]].to_string(float_format='{:.4f}'.format))
print('스프레드(1~28일) {:.1f}%p   (기대 48.6)'.format(
    (_g.loc[28, 'D1'] - _g.loc[1, 'D1']) * 100))

# ── 3) 드리프트 — 측정 불가 집단이 평가 기간에 오르는가 ─────────────────
_p = np.where(base['day'] <= TRAIN_LAST, '(1) 학습', '(2) 평가')
_d = base.pivot_table(index='no_habit', columns=_p, values='d1', aggfunc='mean')
_d['차이(%p)'] = (_d.iloc[:, 1] - _d.iloc[:, 0]) * 100
print('\n습관 측정 여부별 드리프트 (기대: 미측정 0.580→0.666 +8.64%p / 측정 +0.41%p):')
print(_d.to_string(float_format='{:.4f}'.format))

2월 로그 중 사용 행 비중 0.969   (기대 0.969)
측정 창 활동 유저 비중   0.864   (기대 0.864)
기준일 행 중 측정 불가   0.0376  (기대 0.0376)

활동일 수별 D1 (기대: 1일 0.494 → 28일 0.968):
                행     D1
act_days                
0          669813 0.6346
1          150728 0.4935
4          202822 0.4919
7          267643 0.5284
14         464229 0.6592
21         764281 0.8036
27        1478370 0.9402
28        2097468 0.9677
스프레드(1~28일) 47.4%p   (기대 48.6)

습관 측정 여부별 드리프트 (기대: 미측정 0.580→0.666 +8.64%p / 측정 +0.41%p):
          (1) 학습  (2) 평가  차이(%p)
no_habit                        
False     0.7853  0.7894  0.4102
True      0.5797  0.6660  8.6376


## 8. 저장

In [35]:
# ── 1) 모델에 넣을 변수 목록을 못 박아 둠 ───────────────────────────────
# u·day·d1 은 표에는 있지만 단서로 쓰지 않음. 특히 day(달력 날짜)를 넣으면
# "3월 하순은 낮다"를 모델이 외워서 찾아야 할 하락을 삼킴 (계획서 4.2 각주 1)
FEATURES = [
    'days_since_signup', 'base_dow',                                        # A 시간 구조
    'city', 'age', 'gender', 'channel',                                     # B 유저 특성
    'has_tx', 'payment_method_id', 'payment_plan_days', 'plan_list_price',  # C 구독 상태
    'actual_amount_paid', 'discount', 'is_auto_renew', 'is_cancel', 'days_to_expire',
    'act_days', 'mins_per_day', 'pct_full', 'pct_unique',                   # D 평소 습관
    'days_since_last', 'trend', 'weekend_ratio', 'no_habit',
]
# days_since_first_mar 은 일부러 뺐음 — 표에는 있으나 기본 모델에는 안 넣음.
# 넣고 비교하려면 Modeling.ipynb 에서 FEATURES 에 추가하고 REFIT = True 로 재학습하면 됨
print('설명 변수 {}개'.format(len(FEATURES)))
print('표에만 두고 모델엔 안 넣는 열:', [c for c in base.columns if c not in FEATURES])

# ── 2) 결측 현황 ────────────────────────────────────────────────────────
_na = base[FEATURES].isna().mean()
print('\n결측이 있는 열:')
print((_na[_na > 0] * 100).round(3).to_string())

# ── 3) 저장 ─────────────────────────────────────────────────────────────
base.to_parquet(f'{DATA}/mart.parquet', index=False, compression='zstd')
import os
print('\n저장 완료  data/mart.parquet  {:,} 행 × {} 열  ({:.0f} MB)'.format(
    len(base), base.shape[1], os.path.getsize(f'{DATA}/mart.parquet') / 1e6))

설명 변수 23개
표에만 두고 모델엔 안 넣는 열: ['u', 'day', 'd1', 'days_since_first_mar', 'profile']

결측이 있는 열:
days_since_signup     0.002
payment_method_id     1.517
payment_plan_days     1.517
plan_list_price       1.517
actual_amount_paid    1.517
discount              1.517
is_auto_renew         1.517
is_cancel             1.517
days_to_expire        1.518
mins_per_day          3.760
pct_full              3.759
pct_unique            3.759
days_since_last       3.759
trend                 3.759
weekend_ratio         3.759

저장 완료  data/mart.parquet  17,818,632 행 × 28 열  (425 MB)


## 9. 최종 변수 설명

`data/mart.parquet` — **1,781만 행 x 27열**. 한 행 = 유저 1명 x 기준일 1일.

### 키와 결과 변수 (모델 입력 아님)

| 열 | 형 | 뜻 |
|---|---|---|
| `u` | int32 | 유저 번호 (0 ~ 1,099,986). `uniq` 로 원래 `msno` 를 되찾을 수 있음 |
| `day` | int16 | 기준일 번호 (0 = 3/1 … 29 = 3/30) |
| `d1` | bool | **결과 변수.** 기준일 다음 날에도 음악을 들었나 |

⚠️ **`day` 를 설명 변수로 넣으면 안 됨.** 달력 날짜를 알려주면 모델이 "3월 하순은 낮다"를 외워
**찾아야 할 하락을 삼킴.** 기간(달력) 효과는 일부러 모델에 안 넣고 **편차로 잡는 것**이 이 연구의 설계임.
`day` 는 학습·평가 기간을 나누는 데만 씀.

### A. 시간 구조 (2개)

| 열 | 형 | 뜻 | 범위 |
|---|---|---|---|
| `days_since_signup` | float32 | 기준일 − 가입일 | 0 ~ 4,752 |
| `base_dow` | int8 | 기준일 요일 | 0 = 월 … 6 = 일 |

| `days_since_first_mar` | int16 | 기준일 − 3월 첫 활동일 | 0 ~ 29 — **실험용. `FEATURES` 에서 제외** |

### B. 유저 특성 (4개 + 파생 1개)

| 열 | 형 | 값 |
|---|---|---|
| `city` | category | `unknown (1)` · 상위 5개 도시 · `other` · `unknown` |
| `age` | category | `10s` `20s` `30s` `40+` `unknown` |
| `gender` | category | `male` `female` `unknown` |
| `channel` | category | `3` `4` `7` `9` `other` `unknown` |
| `profile` | category | `complete` `missing` `unknown` — **파생. `FEATURES` 에서 제외** |

### C. 구독 상태 (9개) — 기준일 이전 마지막 거래 기준

| 열 | 형 | 뜻 |
|---|---|---|
| `has_tx` | bool | 기준일까지 거래 기록이 있나. **False = 진짜 무료 유저** |
| `payment_method_id` | category | 결제수단 코드 36종. **문자열 범주**(`'41'`)이고 순서는 숫자 순 — parquet 왕복에서 숫자 범주형은 풀려 버리기 때문임 |
| `payment_plan_days` | float32 | 요금제 기간(일) |
| `plan_list_price` | float32 | 정가 |
| `actual_amount_paid` | float32 | 실제 결제액 |
| `discount` | float32 | 정가 − 실제 결제액 |
| `is_auto_renew` | float32 | 자동연장 (0/1) |
| `is_cancel` | float32 | 그 거래가 취소였나 (0/1) |
| `days_to_expire` | float32 | 구독 만료일 − 기준일. **음수 = 이미 만료** (−813 ~ 3,690, 중앙 21) |

### D. 평소 습관 (8개) — 2/1~2/28 에 한 번 측정한 고정값

| 열 | 형 | 뜻 | 범위 |
|---|---|---|---|
| `act_days` | int16 | 2월 활동일 수 | 0 ~ 28 |
| `mins_per_day` | float32 | 활동한 날의 평균 청취 시간(분) | — |
| `pct_full` | float32 | 끝까지 들은 곡 비율 | 0 ~ 1 |
| `pct_unique` | float32 | 서로 다른 곡 비율 | 0 ~ 1 |
| `days_since_last` | float32 | 2/28 − 2월 마지막 활동일 | 0 ~ 27 |
| `trend` | float32 | 후반 14일 − 전반 14일 활동일 수 | −14 ~ +14 |
| `weekend_ratio` | float32 | 토·일 활동일 ÷ 전체 활동일 | 0 ~ 1 |
| `no_habit` | bool | 2월에 활동이 없어 측정 불가 | — |

---

## 결측과 이상치 처리 기준

### 원칙

| | |
|---|---|
| **결측을 지우지 않음** | 결측이 **무작위가 아님** — 가입 경로 4번은 94%·7번은 87%가 결측인데 3·9번은 52%, 가입 연도 2006~2009년은 2~5%인데 2016~2017년은 90%. 결측 행을 빼면 최근 가입자와 특정 경로가 통째로 사라짐 (7-3절) |
| **범주형은 `unknown` 값으로 남김** | 집단 탐색이 "도시 = unknown" 같은 조건을 만들 수 있어야 함. 결측으로 두면 조건이 안 만들어짐 |
| **연속형은 NaN 으로 둠** | LightGBM 이 결측을 그대로 다룸. 억지로 채우면 없는 값이 생김 |
| **클리핑(값 자르기) 안 함** | LightGBM 은 **순위만** 쓰므로 극단값에 흔들리지 않음 |

### 처리 내역

| 대상 | 양 | 어떻게 | 왜 |
|---|---|---|---|
| **3/31 기준일 행** | 유저 3,907명 | **행 자체를 뺌** | 결과일이 4/1인데 데이터에 4월이 없음. "안 돌아왔다"가 아니라 **"모른다"** 라서 0으로 세면 거짓이 생김 |
| `members` 에 없는 유저 | 40명 / 401행 | B군은 `unknown`, `days_since_signup` 은 **NaN** | 3월 활동 유저 중 40명이 `members` 에 없음 |
| 성별 빈칸 | 65% | `unknown` **범주** | 위 원칙 |
| 나이 10~80세 밖 (0 · 음수 · 2016 등) | 67% | `unknown` **범주** | 0이 미입력, −7168 · 2016 같은 값도 있음 |
| 도시 1번 | 71% | `unknown (1)` **범주** | 입력한 행에서는 1번이 19%뿐이라 "정보 없음"으로 판단 |
| **기준일까지 거래 기록 없음** | 27만 행 (1.5%) | C군 9개 전부 **NaN** + `has_tx = False` | **진짜 무료 유저**(D1 0.492). `transactions.csv`(v1) 병합 전에는 이게 "구독 상태 불명"이라 정상 기간 48%였고 월중에 급변해 가짜 가설을 만들 수 있었음 |
| `membership_expire_date` **2000년 이전** | 거래 9,717건 (0.042%)<br>→ 기준일 행으로는 **103행** | `days_to_expire` **NaN** | 19700101 더미 등 "정보 없음"을 0으로 채운 흔적. 그대로 두면 **−17,239일** 이라는 가짜 값이 됨 |
| **`days_to_expire` 음수** (만료일 < 기준일) | **197,487행 (1.11%)** | **음수 그대로 둠** | **오류가 아님.** 취소했거나 이미 만료된 상태라는 **진짜 정보**. 0으로 누르면 "만료 후 며칠 지났는지"를 잃음 |
| 만료일 먼 미래 (최대 2036년) | 0.33% | **그대로 둠** | 장기 요금제일 수 있고, 트리 모델은 극단값에 안 흔들림 |
| `total_secs` > 86,400초 (하루 초과) | 3,820행 (0.024%) | **그 날만** `mins_per_day` 계산에서 제외 | 하루 24시간을 넘는 청취는 불가능. 다른 변수는 그 날도 그대로 씀 |
| `total_secs` 음수 | **0행** | — | 2월 로그에는 없었음 |
| **2월에 활동 없음** | 유저 13.6% / 행 **3.76%** | `act_days = 0`, 나머지 습관 **NaN**, `no_habit = True` | ① 결측으로는 집단 탐색 조건을 못 만듦 ② 이 집단의 D1이 **활동일이 적은 집단보다 높아서** 연속 변수의 끝 값으로 채우면 안 됨 |

### 남은 한계

| | |
|---|---|
| **습관 미측정 집단의 드리프트** | D1이 학습 0.580 → 평가 0.666 (**+8.64%p**) 로 오름. 이 집단의 **48.8%가 3월 신규**라 `days_since_signup` 이 대부분 설명함. 설명되지 않는 부분은 전체 행의 **1.9%** |
| **구독 연차의 좌측 절단** | 거래 데이터가 2015-01 에서 시작해 그 이전 가입자의 구독 이력을 모름 (25%). C 후보 변수로 넣을지는 미정 |

---

## 누수 점검 요약

| 변수 묶음 | 언제 정보인가 | 확인 방법 |
|---|---|---|
| A 시간 | 가입일은 과거, 기준일 요일은 t 자체 | 설계상 자명 |
| B 특성 | `members` 스냅샷 (가입 시점 정보) | `expiration_date` 가 있는 v1 `members` 는 추출 시점 불명이라 **쓰지 않음** |
| C 구독 | **기준일 t 이하**의 마지막 거래 | 6절 끝의 `assert` — 붙은 거래의 거래일이 전부 기준일 이하임을 확인 |
| D 습관 | **2월** (분석 기간 전) | 측정 창이 기준일 범위(3월)와 겹치지 않음 |
| `d1` | **t+1** | 결과 변수라 미래인 것이 맞음. 설명 변수에는 안 씀 |

`is_churn`(공식 이탈 라벨)은 **쓰지 않음** — 구독 기준 라벨이라 청취 활동 리텐션과 지표가 다르고,
3월 말 이후 정보라 누수임.